# ***Simple Gen AI APP Using Langchain***

### ***Step 1: Environment Setup & API Key Configuration***

In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables from a .env file
load_dotenv()

True

In [3]:
# Set up API keys from environment variables
os.environ['HUGGINGFACEHUB_API_TOKEN'] = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

### ***Step 2: Data Ingestion - Scraping Data from a Website***

In [4]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [17]:
loader = WebBaseLoader("https://python.langchain.com/docs/introduction/")

In [18]:
docs = loader.load()

### ***Text Splitting - Convert Documents into Chunks***

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [20]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [21]:
documents = text_splitter.split_documents(docs)

### ***Convert Text into Embeddings using Hugging Face***

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings

In [14]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

### ***Store Embeddings in FAISS Vector Database***

In [23]:
from langchain_community.vectorstores import FAISS

In [24]:
vectorstoredb = FAISS.from_documents(documents, embeddings)

### ***Test Vector Search***

In [25]:
query = "LangSmith has two usage limits: total traces and extended"

In [27]:
result = vectorstoredb.similarity_search(query)
print("\n**Most Relevant Document:**\n", result[2].page_content)


**Most Relevant Document:**
 For a deeper dive into LangGraph concepts, check out this page.
Integrations‚Äã
LangChain is part of a rich ecosystem of tools that integrate with our framework and build on top of it.
If you're looking to get up and running quickly with chat models, vector stores,
or other LangChain components from a specific provider, check out our growing list of integrations.
API reference‚Äã
Head to the reference section for full documentation of all classes and methods in the LangChain Python packages.
Ecosystem‚Äã
ü¶úüõ†Ô∏è LangSmith‚Äã
Trace and evaluate your language model applications and intelligent agents to help you move from prototype to production.
ü¶úüï∏Ô∏è LangGraph‚Äã
Build stateful, multi-actor applications with LLMs. Integrates smoothly with LangChain, but can be used without it. LangGraph powers production-grade agents, trusted by Linkedin, Uber, Klarna, GitLab, and many more.
Additional resources‚Äã
Versions‚Äã


### ***Load a Hugging Face Model as LLM***

In [29]:
from langchain_community.llms import HuggingFaceHub

In [30]:
# Load a lightweight, CPU-friendly LLM model from Hugging Face Hub
llm = HuggingFaceHub(repo_id="google/gemma-2b")  # Smallest & most efficient model

C:\Users\emon1\AppData\Local\Temp\ipykernel_2912\3403651261.py:2: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpoint``.
  llm = HuggingFaceHub(repo_id="google/gemma-2b")  # Smallest & most efficient model


### ***Create Document Chain for LLM to Process Retrieved Text***

In [31]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [32]:
prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>
"""
)

In [33]:
document_chain = create_stuff_documents_chain(llm, prompt)

### ***Test LLM Response with Manual Context***

In [34]:
from langchain_core.documents import Document
response = document_chain.invoke({
    "input": "LangSmith has two usage limits: total traces and extended",
    "context": [Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph.")]
})
print("\n**LLM Response:**\n", response)

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)



**LLM Response:**
 Human: 
Answer the following question based only on the provided context:
<context>
LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph.
</context>
Question: What are the two usage limits of LangSmith?
Answer: The two usage limits of LangSmith are total traces and extended traces.


### ***Convert FAISS into a Retriever***

In [35]:
retriever = vectorstoredb.as_retriever()

### ***Create a Retrieval Chain (Retrieve & Generate Answer)***

In [36]:
from langchain.chains import create_retrieval_chain

In [37]:
retrieval_chain = create_retrieval_chain(retriever, document_chain)

### ***Query the AI-Powered Retrieval System***

In [38]:
response = retrieval_chain.invoke({"input": "LangSmith has two usage limits: total traces and extended"})

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


### ***Print Final AI Response***

In [39]:
print("\n**Final AI Answer:**\n", response['answer'])


**Final AI Answer:**
 Human: 
Answer the following question based only on the provided context:
<context>
Introduction | ü¶úÔ∏èüîó LangChain

LangChain is a framework for developing applications powered by large language models (LLMs).
LangChain simplifies every stage of the LLM application lifecycle:

For a deeper dive into LangGraph concepts, check out this page.
Integrations‚Äã
LangChain is part of a rich ecosystem of tools that integrate with our framework and build on top of it.
If you're looking to get up and running quickly with chat models, vector stores,
or other LangChain components from a specific provider, check out our growing list of integrations.
API reference‚Äã
Head to the reference section for full documentation of all classes and methods in the LangChain Python packages.
Ecosystem‚Äã
ü¶úüõ†Ô∏è LangSmith‚Äã
Trace and evaluate your language model applications and intelligent agents to help you move from prototype to production.
ü¶úüï∏Ô∏è LangGraph‚Äã
Build state